In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification,  get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, TensorDataset, random_split
import os
import time
import numpy as np
import torch
import gc


import copy # Add this line to import the copy module
import sys

from datetime import datetime
import pytz

from data_loader import generate_topology
from main_transformer import run_simulation_transformer
# from data_loader import set_seed # Comment out or remove this line to use the custom set_seed below
from data_loader import distribute_data
from data_loader import get_data
from theoretical_intensity import calculate_theoretical_intensity

from data_loader import set_seed

original_stdout = sys.stdout
original_stderr = sys.stderr
import sys 

class DualLogger(object):
    def __init__(self, file_path):
        self.terminal = sys.stdout
        self.log = open(file_path, "a", encoding='utf-8')

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

    def isatty(self):
        return self.terminal.isatty()

    def close(self):
        if self.log:
            self.log.close()
def deep_clean():
    # 1. 
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    # 2.
    gc.collect()
NUM_CLIENTS = 20
MALICIOUS_RATIO = 0.3
GLOBAL_ROUNDS = 15
# 1. Data & Topology

MODEL_CHECKPOINT = "distilbert-base-uncased"
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
train_ds, test_ds = get_data(
    dataset_name='pubmed',
    tokenizer=TOKENIZER
)
test_loader = DataLoader(test_ds, batch_size=256)
client_datasets = distribute_data(train_ds, NUM_CLIENTS)
#client_datasets = distribute_data_one_class(train_ds, NUM_CLIENTS)
topology_type = 'scale_free'
G = generate_topology(NUM_CLIENTS, topology_type)
neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}
placement_strategy='Topology-Aware'
#
num_mal = int(NUM_CLIENTS * MALICIOUS_RATIO)


bf = 0.5
intensity = 0.02





# ==========================================
# 1. Experiment settup
# ==========================================
BOOST_FACTORS = 2
NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
bf = 1.5
intensity = 0.02
norm_factor = 15
placement_strategy = 'Topology-Aware'

import pytz
from datetime import datetime



MALICIOUS_RATIOS = [0.25]

TOPOLOGY_TYPES = [ 'random_regular','scale_free']
MALICIOUS_RATIOS = [0.3, 0.1, 0.2]
DEFENSE_RATIOS = [ 0.2]
SEEDS = [1, 2, 3]
MECHANISMS =  ['FedAvg']
NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
def_ratio = 0
ratio = 0.3
current_seed = 0
topo_type = 'scale_free'
mech = 'FedAvg'
NORMFACTORS = [15,10,8,5,3,1]
SAVE_PATH = ''
for norm_factor in NORMFACTORS:
  for current_seed in range(3):
      csv_filename = f"intensity_Transformer_{topo_type}_MR{ratio}_INTENSITY{intensity}_norm_factor{norm_factor}_{mech}_seed{current_seed}.csv"

      full_save_path = os.path.join(SAVE_PATH, csv_filename)
      log_filename = f"Log_Transformer_{topo_type}_MR{ratio}_DR{def_ratio}_{mech}_seed{current_seed}.txt"

      full_save_path = os.path.join(SAVE_PATH, csv_filename)
      log_full_path = os.path.join(SAVE_PATH, log_filename)

      all_results = []


      logger = DualLogger(log_full_path)
      sys.stdout = logger
      sys.stderr = logger

      print(f"\n{'='*60}")
      print(f"⏰ : {time.strftime('%Y-%m-%d %H:%M:%S')}")
      print(f"📡 : Topo={topo_type}, Mal={ratio}, Def={def_ratio}, Seed={current_seed},norm_factor={norm_factor}")
      print(f"{'='*60}")

      set_seed(current_seed)


      G = generate_topology(NUM_CLIENTS, topo_type)
      neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}


      malicious_clients= list(np.random.choice(range(NUM_CLIENTS), int(NUM_CLIENTS*ratio), replace=False))
      defense_nodes = list()
      theo_intensities = calculate_theoretical_intensity(
          neighbors, malicious_clients, NUM_CLIENTS, BOOST_FACTORS, lambda_benign=0.3
      )

      start_tick = time.time()

      ctx = mp.get_context('spawn')

      with ProcessPoolExecutor(max_workers=1, mp_context=ctx) as executor:

          future = executor.submit(
              run_simulation_transformer,
              current_seed, NUM_CLIENTS, defense_nodes, malicious_clients,
              G, neighbors, client_datasets, test_ds,
              #
              mechanism=mech, bf=bf, intensity=intensity,
              debug=False, GLOBAL_ROUNDS=GLOBAL_ROUNDS,
              norm_factor=norm_factor, epochs=1
          )

          #
          _, _, accs, asrs = future.result()
      print(f"🚀 Running: {mech}")
      end_tick = time.time()
      duration_sec = round(end_tick - start_tick, 2)

      #
      result_entry_base = {
          'seed': current_seed, 'mechanism': mech, 'malicious_ratio': ratio,
          'defense_ratio': def_ratio, 'topology': topo_type, 'norm_factor': norm_factor,
          'duration_sec': duration_sec
      }

      for i in range(NUM_CLIENTS):
          client_row = copy.deepcopy(result_entry_base)
          client_row.update({
              'client_id': i, 'final_acc': accs[i], 'final_asr': asrs[i],
              'theo_intensity': theo_intensities[i] if i < len(theo_intensities) else 0.0,
              'node_type': 'MAL' if i in malicious_clients else ('DEF' if i in defense_nodes else 'BEN'),
              'neighbors': str(list(neighbors.get(i, [])))

          })
          all_results.append(client_row)

      #
      pd.DataFrame(all_results).to_csv(full_save_path, index=False)

 
      #

      sys.stdout = logger
      sys.stdout = original_stdout
      sys.stderr = original_stderr
      logger.close()

      deep_clean()


print(f"\n🎉Experiments compeleted: {SAVE_PATH}")

In [ ]:
import os
import glob
import pandas as pd
from scipy.stats import kendalltau  

# ==========================================
# 
# ==========================================
print(f"\n{'='*40}")
print(f"{'='*40}")
SAVE_DIR =''
fedavg_pattern = os.path.join(SAVE_DIR, "intensity_Transformer*.csv")
fedavg_files = glob.glob(fedavg_pattern)

correlation_results = []
all_fedavg_nodes = [] 

for f in fedavg_files:
    df = pd.read_csv(f)

    # 1. 
    df_target = df.copy()

    # 2. 
    df_target['final_asr'] = pd.to_numeric(df_target['final_asr'], errors='coerce')
    df_target['theo_intensity'] = pd.to_numeric(df_target['theo_intensity'], errors='coerce')

    # 3. 
    df_target = df_target.dropna(subset=['final_asr', 'theo_intensity'])

    if len(df_target) > 1: 
        # 4. 
        topo = df_target['topology'].iloc[0] if 'topology' in df_target.columns else "Unknown"
        mr = df_target['malicious_ratio'].iloc[0] if 'malicious_ratio' in df_target.columns else "Unknown"
        seed = df_target['seed'].iloc[0] if 'seed' in df_target.columns else "Unknown"

        # 5. 
        tau, p_value = kendalltau(df_target['final_asr'], df_target['theo_intensity'])

        # 6. 
        correlation_results.append({
            'File': os.path.basename(f),
            'Topology': topo,
            'Mal_Ratio': mr,
            'Seed': seed,
            'Nodes_Count': len(df_target),
            'Kendall_Tau': tau,
            'P_Value': p_value,
            'Is_Significant': 'Yes' if p_value < 0.05 else 'No'
        })

        all_fedavg_nodes.append(df_target)



In [ ]:
# Kendall's tau
if correlation_results:
    df_results = pd.DataFrame(correlation_results)


    df_results = df_results.sort_values(by='Kendall_Tau', ascending=False)


    avg_tau = df_results['Kendall_Tau'].mean()
    significant_count = (df_results['P_Value'] < 0.05).sum()
    total_count = len(df_results)


In [ ]:
#Figure 1a
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# ==========================================
# 0. 
# ==========================================
TARGET_NORM_FACTOR = 10
TARGET_MECH = "FedAvg"
FOLDER_PATH = ''

file_pattern = os.path.join(FOLDER_PATH, f"*norm_factor{TARGET_NORM_FACTOR}_*.csv")
csv_files = glob.glob(file_pattern)


df_list = [pd.read_csv(f) for f in csv_files]
df_all = pd.concat(df_list, ignore_index=True)

if 'seed' not in df_all.columns:
    df_all['seed'] = np.repeat(np.arange(len(csv_files)), len(df_list[0]))

df_all['intensity_rank'] = df_all.groupby(['seed', 'node_type'])['theo_intensity'].rank(method='first')

df_avg = df_all.groupby(['node_type', 'intensity_rank']).agg({
    'theo_intensity': 'mean',
    'final_asr': 'mean',
    'final_acc': 'mean'
}).reset_index()


df_avg['final_asr'] = df_avg['final_asr']
df_avg['final_acc'] = df_avg['final_acc']

df_avg = df_avg.sort_values(by='theo_intensity').reset_index(drop=True)

df_avg['client_label'] = df_avg['node_type'] + '\n(Rank ' + df_avg['intensity_rank'].astype(int).astype(str) + ')'

colors_theo = ['#1F77B4' if nt != 'MAL' else '#002E5D' for nt in df_avg['node_type']]
colors_asr  = ['#FF7F0E' if nt != 'MAL' else '#D62728' for nt in df_avg['node_type']]

# ==========================================
# 
# ==========================================
x = np.arange(len(df_avg))
width = 0.35

fig, ax1 = plt.subplots(figsize=(16, 7))


rects1 = ax1.bar(x - width/2, df_avg['theo_intensity'], width, color=colors_theo, edgecolor='white')
ax1.set_ylabel('Theoretical Diffusion Bound', color='#1F77B4', fontsize=14, fontweight='bold')
ax1.tick_params(axis='y', labelcolor='#1F77B4', labelsize=12)
ax1.set_xlabel('Nodes Ranked by Theoretical Diffusion Bound', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(df_avg['client_label'], fontsize=11, rotation=45, ha='right')


for text in ax1.get_xticklabels():
    if 'MAL' in text.get_text():
        text.set_color('#D62728')
        text.set_fontweight('bold')

ax1.grid(True, axis='y', linestyle='--', alpha=0.4)


ax2 = ax1.twinx()
rects2 = ax2.bar(x + width/2, df_avg['final_asr'], width, color=colors_asr, edgecolor='white')

asr_max = df_avg['final_asr'].max()
is_percentage = asr_max > 1.5
ax2.set_ylabel(f'ASR {"(%)" if is_percentage else ""}', color='#D62728', fontsize=14, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#D62728', labelsize=12)
ax2.set_ylim(0, 105 if is_percentage else 1.05)


legend_elements = [
    Patch(facecolor='#1F77B4', edgecolor='w', label='Theoretical Diffusion Bound (BEN)'),
    Patch(facecolor='#002E5D', edgecolor='w', label='Theoretical Diffusion Bound (MAL)'),
    Patch(facecolor='#FF7F0E', edgecolor='w', label='ASR (BEN)'),
    Patch(facecolor='#D62728', edgecolor='w', label='ASR (MAL)')
]
ax1.legend(handles=legend_elements, loc='upper left', fontsize=11, frameon=True, shadow=True)

plt.title(fr'Comparison of Theoretical Diffusion Bound vs. ASR ($b_f$ = {TARGET_NORM_FACTOR})',
          fontsize=16, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

In [ ]:
#Figure 1b
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# 
# ==========================================

df_list = [pd.read_csv(f) for f in csv_files]
df_all = pd.concat(df_list, ignore_index=True)

df_benign = df_all[df_all['node_type'] == 'BEN']
df_benign['final_asr'] = df_benign['final_asr']*100
df_benign['final_acc'] = df_benign['final_acc']*100

df_phase = df_benign.groupby('norm_factor')[['final_asr', 'final_acc']].mean().reset_index()
df_phase = df_phase.sort_values(by='norm_factor').reset_index(drop=True)

print("\n聚合后的相变数据：")
print(df_phase)

# ==========================================
# 
# ==========================================
fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.set_xlabel(fr'$b_f$', fontsize=14, fontweight='bold')
ax1.set_xticks(df_phase['norm_factor'])
ax1.set_xticklabels(df_phase['norm_factor'], fontsize=9)


color_asr = '#D62728'
ax1.set_ylabel('Average ASR on Benign Nodes (%)', color=color_asr, fontsize=14, fontweight='bold')
line1 = ax1.plot(df_phase['norm_factor'], df_phase['final_asr'],
                 color=color_asr, marker='o', linewidth=3, markersize=8, label='ASR')
ax1.tick_params(axis='y', labelcolor=color_asr, labelsize=12)
ax1.set_ylim(-5, 105) 


ax1.grid(True, linestyle='--', alpha=0.6)

ax2 = ax1.twinx()
color_acc = '#1F77B4'
ax2.set_ylabel('Average ACC on Benign Nodes (%)', color=color_acc, fontsize=14, fontweight='bold')
line2 = ax2.plot(df_phase['norm_factor'], df_phase['final_acc'],
                 color=color_acc, marker='s', linestyle='--', linewidth=3, markersize=8, label='ACC')
ax2.tick_params(axis='y', labelcolor=color_acc, labelsize=12)
ax2.set_ylim(-5, 105)

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='center right', fontsize=12, frameon=True, shadow=True)

plt.title(fr'ACC, ASR vs. $b_f$', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()